In [1]:
# Verify sample shapes
import numpy as np

X_all = np.load('X_all.npy')
Y_all = np.load('Y_all.npy')
print(X_all.shape)
print(Y_all.shape)
# print(X_all[1, :, :])

import numpy as np

X_all = np.load('X_all.npy')

for i in range(X_all.shape[1]):
    print(f'channel {i+1}: min={X_all[:, i, :].min()}, max={X_all[:, i, :].max()}')



(512000, 4, 64)
(512000, 2, 64)
channel 1: min=0.7781354521047212, max=1285.2102012252547
channel 2: min=5.929800588334302, max=84.06759196305127
channel 3: min=0.7781374003123184, max=1285.0426150159642
channel 4: min=5.938388247787728, max=84.06415524363595


In [ ]:
# Main code
import torch
import torch.nn as nn
import argparse
import time
import copy
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np



import torch
import torch.nn as nn
import copy

# ================= Basic modules =================
class Norm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(d_model, eps=eps)

    def forward(self, x):
        return self.norm(x)

class PositionalEncoder(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = Norm(d_model)
        self.norm2 = Norm(d_model)

    def forward(self, x, mask=None):
        attn_out, _ = self.attn(x, x, x, key_padding_mask=mask)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x

def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

# ================= Transformer Encoder =================
class Encoder(nn.Module):
    def __init__(self, in_dim, d_model, N, heads, dropout):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, d_model)
        self.pe = PositionalEncoder(d_model, dropout)
        self.layers = get_clones(EncoderLayer(d_model, heads, dropout), N)
        self.norm = Norm(d_model)

    def forward(self, src, mask=None):
        x = self.input_proj(src)
        x = self.pe(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# ================= Encoder-Only Transformer =================
class EncoderOnlyTransformer(nn.Module):
    def __init__(self, d_model, N, heads, dropout, src_dim=4, out_dim=2):
        super().__init__()
        self.encoder = Encoder(
            in_dim=src_dim,
            d_model=d_model,
            N=N,
            heads=heads,
            dropout=dropout
        )
        # Add one hidden layer to the output head to improve nonlinear mapping
        self.out = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, out_dim)
        )

    def forward(self, src, mask=None):
        """
        src: (batch, seq_len, in_dim)
        return: (batch, seq_len, out_dim)
        """
        x = self.encoder(src, mask)
        return self.out(x)

# ================= Model initialization function =================
def get_model(opt):
    model = EncoderOnlyTransformer(
        d_model=opt.d_model,
        N=opt.n_layers,
        heads=opt.heads,
        dropout=opt.dropout,
        src_dim=opt.src_dim if hasattr(opt, 'src_dim') else 4,
        out_dim=opt.out_dim if hasattr(opt, 'out_dim') else 2
    ).to(opt.device)

    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return model




def train_model(model, opt, loader):
    model.train()
    loss_fn = nn.MSELoss()

    cosine_scheduler = None

    for epoch in range(opt.epochs):
        total_loss = 0.0

        for src, trg in loader:
            src, trg = src.to(opt.device), trg.to(opt.device)

            opt.optimizer.zero_grad()
            preds = model(src)
            loss = loss_fn(preds, trg)
            loss.backward()
            opt.optimizer.step()

            total_loss += loss.item()

        # Use the current StepLR for the first 100 epochs
        if epoch < opt.switch_epoch:
            opt.step_scheduler.step()

        # Switch to CosineAnnealingLR starting from epoch 101
        else:
            if cosine_scheduler is None:
                cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    opt.optimizer,
                    T_max=opt.epochs - opt.switch_epoch,
                    eta_min=opt.eta_min
                )
            cosine_scheduler.step()

        lr = opt.optimizer.param_groups[0]['lr']

        print(
            f"Epoch [{epoch+1}/{opt.epochs}], "
            f"Loss: {total_loss/len(loader):.6f}, "
            f"LR: {lr:.6e}"
        )

# def train_model(model, opt, loader):
#     model.train()
#     loss_fn = nn.MSELoss()

#     for epoch in range(opt.epochs):
#         total_loss = 0.0

#         for src, trg in loader:
#             src, trg = src.to(opt.device), trg.to(opt.device)

#             opt.optimizer.zero_grad()
#             preds = model(src)
#             loss = loss_fn(preds, trg)
#             loss.backward()
#             opt.optimizer.step()

#             total_loss += loss.item()

#         opt.scheduler.step()
#         lr = opt.optimizer.param_groups[0]['lr']

#         print(
#             f"Epoch [{epoch+1}/{opt.epochs}], "
#             f"Loss: {total_loss/len(loader):.6f}, "
#             f"LR: {lr:.6e}"
#         )


class Seq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        """
        X: Tensor, shape = (N, 4, 64)
        Y: Tensor, shape = (N, 2, 64)
        """


        self.X = torch.FloatTensor(X)
        self.Y = torch.FloatTensor(Y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]



def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False




if __name__ == "__main__":
    
    set_seed(seed=42)
    parser = argparse.ArgumentParser()
    parser.add_argument('-epochs', type=int, default=300)
    parser.add_argument('-d_model', type=int, default=192)
    parser.add_argument('-n_layers', type=int, default=3)
    parser.add_argument('-heads', type=int, default=8)
    parser.add_argument('-dropout', type=float, default=0.03)
    parser.add_argument('-batchsize', type=int, default=64)
    parser.add_argument('-lr', type=float, default=3e-4)
    parser.add_argument('-no_cuda', action='store_true')

    # StepLR parameters for the first phase
    parser.add_argument('-lr_step', type=int, default=50)
    parser.add_argument('-lr_gamma', type=float, default=0.9)

    # Parameters for switching to cosine annealing
    parser.add_argument('-switch_epoch', type=int, default=100)
    parser.add_argument('-eta_min', type=float, default=1e-5)
    

    opt = parser.parse_args(args=[])
    opt.device = 'cuda' if (not opt.no_cuda and torch.cuda.is_available()) else 'cpu'

    # X_all = np.load('X_all.npy')
    # Y_all = np.load('Y_all.npy')
    
    # # ===== 1. Take logarithms =====
    # X_log = np.log(X_all)
    # Y_log = np.log(Y_all)

    # # ===== 2. Compute min / max from the data itself =====
    # X_log_min = X_log.min()
    # X_log_max = X_log.max()

    # Y_log_min = Y_log.min()
    # Y_log_max = Y_log.max()

    # # ===== 3. Log-MinMax normalization =====
    # X_norm = (X_log - X_log_min) / (X_log_max - X_log_min)
    # Y_norm = (Y_log - Y_log_min) / (Y_log_max - Y_log_min)

    

    # # (N, C, L) -> (N, L, C)
    # X_norm = np.transpose(X_norm, (0, 2, 1))
    # Y_norm = np.transpose(Y_norm, (0, 2, 1))


    # dataset = Seq2SeqDataset(X_norm, Y_norm)
    # loader = DataLoader(dataset, batch_size=opt.batchsize, shuffle=True)

    X_all = np.load('X_all.npy')
    Y_all = np.load('Y_all.npy')

    # ===== 1. Take logarithms =====
    X_log = np.log(X_all)
    Y_log = np.log(Y_all)

    # ===== 2. Compute min / max for X by group =====
    # Channels 1 and 3: apparent resistivity
    X_rho = X_log[:, [0, 2], :]
    X_rho_min = X_rho.min()
    X_rho_max = X_rho.max()

    # Channels 2 and 4: phase
    X_phase = X_log[:, [1, 3], :]
    X_phase_min = X_phase.min()
    X_phase_max = X_phase.max()

    # Compute Y globally as before
    Y_log_min = Y_log.min()
    Y_log_max = Y_log.max()

    # ===== 3. Grouped Log-MinMax normalization =====
    X_norm = np.empty_like(X_log)

    # Channels 1 and 3 share one set of normalization parameters
    X_norm[:, [0, 2], :] = (X_log[:, [0, 2], :] - X_rho_min) / (X_rho_max - X_rho_min)

    # Channels 2 and 4 share one set of normalization parameters
    X_norm[:, [1, 3], :] = (X_log[:, [1, 3], :] - X_phase_min) / (X_phase_max - X_phase_min)

    # Keep the original global normalization method for Y
    Y_norm = (Y_log - Y_log_min) / (Y_log_max - Y_log_min)

    # (N, C, L) -> (N, L, C)
    X_norm = np.transpose(X_norm, (0, 2, 1))
    Y_norm = np.transpose(Y_norm, (0, 2, 1))

    dataset = Seq2SeqDataset(X_norm, Y_norm)
    loader = DataLoader(dataset, batch_size=opt.batchsize, shuffle=True)



    model = get_model(opt)


    opt.optimizer = torch.optim.AdamW(
        model.parameters(), lr=opt.lr, weight_decay=1e-5
    )

    opt.step_scheduler = torch.optim.lr_scheduler.StepLR(
        opt.optimizer,
        step_size=opt.lr_step,
        gamma=opt.lr_gamma
    )


    # opt.optimizer = torch.optim.AdamW(
    #     model.parameters(), lr=opt.lr, weight_decay=1e-5
    # )

    # opt.scheduler = torch.optim.lr_scheduler.StepLR(
    #     opt.optimizer,
    #     step_size=opt.lr_step,
    #     gamma=opt.lr_gamma
    # )

    train_model(model, opt, loader)

torch.save(model.state_dict(), 'new003.pth') # d_model n_layers heads batch size sample count

Epoch [1/300], Loss: 0.006651, LR: 3.000000e-04
Epoch [2/300], Loss: 0.001427, LR: 3.000000e-04
Epoch [3/300], Loss: 0.000982, LR: 3.000000e-04
Epoch [4/300], Loss: 0.000773, LR: 3.000000e-04
Epoch [5/300], Loss: 0.000653, LR: 3.000000e-04
Epoch [6/300], Loss: 0.000569, LR: 3.000000e-04
Epoch [7/300], Loss: 0.000512, LR: 3.000000e-04
Epoch [8/300], Loss: 0.000467, LR: 3.000000e-04
Epoch [9/300], Loss: 0.000431, LR: 3.000000e-04
Epoch [10/300], Loss: 0.000410, LR: 3.000000e-04
Epoch [11/300], Loss: 0.000383, LR: 3.000000e-04
Epoch [12/300], Loss: 0.000367, LR: 3.000000e-04
Epoch [13/300], Loss: 0.000348, LR: 3.000000e-04
Epoch [14/300], Loss: 0.000337, LR: 3.000000e-04
Epoch [15/300], Loss: 0.000320, LR: 3.000000e-04
Epoch [16/300], Loss: 0.000308, LR: 3.000000e-04
Epoch [17/300], Loss: 0.000304, LR: 3.000000e-04
Epoch [18/300], Loss: 0.000293, LR: 3.000000e-04
Epoch [19/300], Loss: 0.000283, LR: 3.000000e-04
Epoch [20/300], Loss: 0.000277, LR: 3.000000e-04
Epoch [21/300], Loss: 0.00026